# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

*Look before deciding: distributions of your key fields. Note the heavy tails.*

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Key distributions for signal audit
print('--- Distributions summary ---')
print('impressions_90d skew:', round(df['impressions_90d'].skew(), 2))
print('clicks_90d skew:', round(df['clicks_90d'].skew(), 2))
print('sessions_90d skew:', round(df['sessions_90d'].skew(), 2))
print('ctr skew:', round(df['ctr'].skew(), 2))
print('engagement_rate skew:', round(df['engagement_rate'].skew(), 2))
print('avg_position skew:', round(df['avg_position'].skew(), 2))
print('avg_position = 0 rows:', (df['avg_position'] == 0).sum())
print('ctr range:', round(df['ctr'].min(), 2), 'to', round(df['ctr'].max(), 2))
print('engagement_rate range:', round(df['engagement_rate'].min(), 2), 'to', round(df['engagement_rate'].max(), 2))

## 2. Signal test #1 / #2 / #3 (verdict each)

*Three safe signals, each with a mini-test and a verdict: CONFIRMED / OPPOSITE / MIXED / FALSE.*

In [ ]:
print()
# Signal 1: has_clicks
has_clicks_decline = df[df['has_clicks'] == 1]['is_declining_label'].mean()
no_clicks_decline = df[df['has_clicks'] == 0]['is_declining_label'].mean()
print('Signal 1 - Decline rate with clicks:', round(has_clicks_decline, 4))
print('Signal 1 - Decline rate without clicks:', round(no_clicks_decline, 4))
v1 = 'CONFIRMED' if no_clicks_decline > has_clicks_decline else 'OPPOSITE' if no_clicks_decline < has_clicks_decline else 'MIXED'
print('Signal 1 Verdict:', v1)

print()
# Signal 2: engagement_rate
median_er = df['engagement_rate'].median()
high_er_decline = df[df['engagement_rate'] > median_er]['is_declining_label'].mean()
low_er_decline = df[df['engagement_rate'] <= median_er]['is_declining_label'].mean()
print('Signal 2 - Decline rate high engagement:', round(high_er_decline, 4))
print('Signal 2 - Decline rate low engagement:', round(low_er_decline, 4))
v2 = 'CONFIRMED' if high_er_decline < low_er_decline else 'OPPOSITE' if high_er_decline > low_er_decline else 'MIXED'
print('Signal 2 Verdict:', v2)
print()
# Signal 3: ai_traffic_pct buckets
df['ai_bucket'] = pd.cut(df['ai_traffic_pct'], bins=[-1, 20, 40, 60, 100, 101], labels=['0-20','20-40','40-60','60-100','100+'])
ai_decline = df.groupby('ai_bucket')['is_declining_label'].mean().round(4)
print('Signal 3 - Decline rate by AI traffic %:')
print(ai_decline)
print('Verdict: FALSE' if ai_decline.std() < 0.05 else 'MIXED')

## 3. The flag-linked test

*Pick a signal one of FlyRank's real flags relies on. Does the data support the rule's assumption?*

In [ ]:
print()
# Flag-linked test: freshness_tier
ft_decline = df[df['freshness_tier'] == 'never']['is_declining_label'].mean()
other_decline = df[df['freshness_tier'] != 'never']['is_declining_label'].mean()
print('Signal - Decline rate never updated:', round(ft_decline, 4))
print('Signal - Decline rate had update:', round(other_decline, 4))
if ft_decline > other_decline:
    print('Interpretation: stale content associated with higher decline (confirms stale = decline assumption)')
else:
    print('Interpretation: stale content NOT associated with higher decline (assumption not supported)')

## 4. What this means in practice

*Two or three sentences: what a content team should take from this.*

In [ ]:
# Summary of all verdicts
print('=== Signal Audit Summary ===')
print('Signal 1 (has_clicks):', v1)
print('Signal 2 (engagement_rate):', v2)
print('Signal 3 (ai_traffic_pct):', 'FALSE' if ai_decline.std() < 0.05 else 'MIXED')
print()
- If has_clicks=0 strongly predicts decline: prioritize zero-click pages for optimization or pruning
- If high engagement_rate protects against decline: invest in quality engagement pages first
- If ai_traffic_pct shows no reliable signal: AI referral % is not a good performance predictor; focus on GSC/GA4 metrics
- If freshness_tier=never predicts higher decline: maintain a content refresh schedule; stale content tends to decline

## Self-check

Before you submit, confirm each line honestly:
- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.